In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo 

server_timestamp = datetime.now(ZoneInfo("New_York")).strftime("%Y-%m-%d %H:%M:%S")
print(server_timestamp)

In [ ]:
# Generating new table in sql db
import sqlite3

conn = sqlite3.connect("habits.db")
cursor = conn.cursor()
cursor.execute('''CREATE TABLE IF NOT EXISTS workouts (
        work_list TEXT)''')
conn.commit()
conn.close()

In [ ]:
import json
import sqlite3
# adding a list of workouts to the table
conn = sqlite3.connect("habits.db")
cursor = conn.cursor()
# intial workout list gained from original google sheets file
workouts = [
    "Biceps seated db curls",
    "Back row machine",
    "Core abdominal machine",
    "Core torso rotation machine",
    "Biceps curl machine",
    "Back ISO rear deltoid machine",
    "Chest press machine",
    "Triceps extension machine",
    "Shoulders db complex",
    "Chest pectoral fly machine",
    "Triceps db skull crushers",
    "Back pull-down machine",
    "Biceps db preacher curls",
    "Chest db bench press",
    "Leg extension machine",
    "Leg curl machine",
    "Hip abduction",
    "Hip adduction",
    "Calf leg press machine",
    "Leg press machine",
    "Biceps standing db curls",
    "Triceps pull-down",
    "Biceps curl machine high",
    "Back rear deltoid machine",
    "Chest smith machine bench",
    "Chest incline press machine",
    "Back extension machine",
    "Core complex",
    "Shoulders press machine",
    "Triceps press machine",
    "Chest push ups",
    "Hip addiction machine",
    "Calf extension machine",
    "Chest bench press",
    "Back assisted pull up machine",
    "Shoulders deltoid raise machine",
    "Trcieps pull-down",
    "other"]

workouts_json = json.dumps(workouts)

cursor.execute('''
            INSERT INTO workouts (work_list)
            VALUES (?)
        ''', (workouts_json,))
conn.commit()
conn.close()

In [ ]:
# Reading in the json from the sql db
import json
import sqlite3

conn = sqlite3.connect("habits.db")
cursor = conn.cursor()
cursor.execute("SELECT work_list FROM workouts")
result = cursor.fetchone() # only fetches one row from db

workout_list = json.loads(result[0])
print(workout_list)  # results back into list

['Bench Press', 'Squats', 'Deadlifts', 'Overhead Press', 'Pull-Ups', 'Push-Ups', 'Lunges', 'Plank', 'Burpees', 'Bicep Curls']


In [ ]:
# Merge exisiting workouts google sheet to sql db

import pandas as pd
from sqlalchemy import create_engine
import pandas as pd
import os

sheet_id = "1TM8IcrqlYRlk8Ggxig1XMGh5e12KzZTGbh41tp6IB6k"
sheet_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

workouts = pd.read_csv(sheet_url)

def df_to_sqlite(df: pd.DataFrame, name: str, db_path: str) -> None:
    # Input validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")
    if not isinstance(name, str):
        raise TypeError("name must be a string")
    if not isinstance(db_path, str):
        raise TypeError("db_path must be a string")

    # Create a SQLite connection string
    engine = create_engine(f"sqlite:///{db_path}")

    # Save the DataFrame to the SQLite database
    df.to_sql(name=name, con=engine, if_exists='replace', index=True)

    # Dispose the engine
    engine.dispose()


df_to_sqlite(workouts, "workouts", "habits.db")